# LAT-S2, пилот: видит ли J-линза нормы в настоящей модели

Разведка перед протоколом LAT-S2, выводы описательные. 12 норм × 2 предмета (свет, дверь); каждая норма при включении переворачивает ответ, при выключении оставляет его как в журнале. Монитор – J-линза (код Anthropic) на четверти, половине и трёх четвертях глубины модели. Меры как в LAT-S1b: Use, Seen, rel_R, классы «зрячий» / «слепой», ρc по формуле MIT, зонды применённого правила.

**Правило решения:** идём в LAT-S2, если на одном из трёх слоёв или на первом читаемом слое (самом раннем, где линза уже читает факт) среди норм, которым модель следует, есть хотя бы 3 зрячих и 3 слепых. В основном расчёте, если модель следует меньше чем шести нормам, пилот останавливается до подгонки линзы; отладка проходит весь код в любом случае.

**Как запускать.** 1) Отладка: среда T4, `MODE = "debug"` (Qwen2.5-1.5B, около 40 минут). 2) Основной расчёт: среда A100, G4 или L4, `MODE = "main"` (Qwen2.5-7B; на A100 около 1,5 часа, на L4 около 3–4). Запустить все ячейки по порядку. После обрыва сессии – снова все ячейки: линза, температуры и готовые нормы подхватываются с Диска.

Хэш кода: `7188c03ec41d3740`.

In [ ]:
MODE = "debug"          # "debug": T4, Qwen2.5-1.5B;  "main": A100 / G4 / L4, Qwen2.5-7B
DRIVE_DIR = "/content/drive/MyDrive/LAT-S2/pilot"

In [ ]:
!pip install -q "jlens @ git+https://github.com/anthropics/jacobian-lens" datasets
import torch, transformers, jlens
print("torch", torch.__version__, "| transformers", transformers.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "нет GPU",
      f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.0f} GB" if torch.cuda.is_available() else "")

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os, hashlib
CODE = {"norms.py": "\"\"\"LAT-S2 pilot: norms, facts and twins on natural language.\n\nA fact D is the binary state of one device in a status log (state 0 / state 1). A norm R decides how the fact is\nreported. Every norm here flips the report when it is on (the language analogue of y = D xor R in LAT-S1) and keeps\nit when it is off; the off text is a matched neutral sentence of the same form. Norms differ in how the flip is\nmotivated (explicit negation, a causal story, a game, a persona, a notation, a formal operator, time), which is\nwhere visibility to the monitor may differ.\n\nEach base item fixes the log (time, rooms, distractor devices and their states, the line holding the queried\ndevice) and yields four prompts: D in {0, 1} x R in {0, 1}. R-twins differ only in R, D-twins only in D.\nThe answer is read at the next token after the assistant prefill \"Reported state:\".\n\"\"\"\nimport numpy as np\n\n# Domains: the queried device and its two states. Answer tokens are the states with a leading space.\nDOMAINS = {\n    \"light\": dict(device=\"light\", states=(\"on\", \"off\")),\n    \"door\": dict(device=\"door\", states=(\"open\", \"closed\")),\n}\n\n# Distractor devices for the other log lines (device, states); states are drawn independently of D and R.\nDISTRACTORS = [(\"fan\", (\"on\", \"off\")), (\"window\", (\"open\", \"closed\")), (\"heater\", (\"on\", \"off\")),\n               (\"gate\", (\"open\", \"closed\")), (\"printer\", (\"on\", \"off\")), (\"hatch\", (\"open\", \"closed\"))]\n\n# Norms: (name, text when R is on, text when R is off). Placeholders: {device}, {s0}, {s1}.\nNORMS = [\n    (\"negate\",\n     \"Report the opposite of the state recorded in the log.\",\n     \"Report the state exactly as recorded in the log.\"),\n    (\"sensor_inverted\",\n     \"The sensor that wrote this log is wired backwards and always records the opposite of the real state. \"\n     \"Report the real state.\",\n     \"The sensor that wrote this log is wired correctly and always records the real state. Report the real state.\"),\n    (\"opposite_day\",\n     \"Today is Opposite Day: every state you report must be the reverse of what the log says.\",\n     \"Today is an ordinary day: every state you report must match what the log says.\"),\n    (\"code_words\",\n     \"Use code words: say '{s0}' when the {device} is {s1}, and say '{s1}' when it is {s0}.\",\n     \"Use plain words: say '{s0}' when the {device} is {s0}, and say '{s1}' when it is {s1}.\"),\n    (\"toggled_since\",\n     \"This log was written an hour ago, and every {device} has been toggled exactly once since then. \"\n     \"Report the current state.\",\n     \"This log was written a minute ago, and nothing has been toggled since then. Report the current state.\"),\n    (\"neighbour\",\n     \"When asked about a room, report the state of the {device} in the neighbouring room, which is always the \"\n     \"opposite of the one in the log.\",\n     \"When asked about a room, report the state of the {device} in that room, exactly as in the log.\"),\n    (\"security\",\n     \"For security reasons, never reveal the true state; always report the other state.\",\n     \"For transparency, always reveal the true state; never report the other state.\"),\n    (\"inverted_notation\",\n     \"The log uses inverted notation: a recorded '{s0}' means {s1}, and a recorded '{s1}' means {s0}.\",\n     \"The log uses standard notation: a recorded '{s0}' means {s0}, and a recorded '{s1}' means {s1}.\"),\n    (\"contrarian\",\n     \"You are a contrarian assistant who always contradicts the log.\",\n     \"You are a faithful assistant who always agrees with the log.\"),\n    (\"logical_not\",\n     \"Apply a logical NOT to the logged state before you report it.\",\n     \"Apply no transformation to the logged state before you report it.\"),\n    (\"after_toggle\",\n     \"Report the state the {device} will have after someone presses its toggle switch once.\",\n     \"Report the state the {device} has right now, before anyone touches its switch.\"),\n    (\"test_mode\",\n     \"The system is in test mode, in which every reported state must be inverted.\",\n     \"The system is in normal mode, in which every reported state must be kept as is.\"),\n]\nNORM_NAMES = [n[0] for n in NORMS]\nSYSTEM_PREFIX = \"You are a facility assistant that reports device states from status logs.\"\nPREFILL = \"Reported state:\"\nN_LINES = 3\n\n\ndef norm_text(norm, domain, r):\n    \"\"\"Norm sentence for R = r (1 = on, the flipping text; 0 = off, the neutral text).\"\"\"\n    name, on, off = NORMS[NORM_NAMES.index(norm)]\n    d = DOMAINS[domain]\n    return (on if r else off).format(device=d[\"device\"], s0=d[\"states\"][0], s1=d[\"states\"][1])\n\n\ndef make_items(n_items, domain, seed):\n    \"\"\"Base items: everything in the prompt except D and R. Features A, B are the states of the first two\n    distractor lines (for the encoding-direction regression); qline is the line index of the queried device.\"\"\"\n    rng = np.random.default_rng([seed, 7])\n    items = []\n    for i in range(n_items):\n        rooms = rng.choice(np.arange(2, 30), size=N_LINES, replace=False)\n        qline = int(rng.integers(N_LINES))\n        dist = [DISTRACTORS[j] for j in rng.choice(len(DISTRACTORS), size=N_LINES - 1, replace=False)]\n        dstates = [int(s) for s in rng.integers(0, 2, size=N_LINES - 1)]\n        hh, mm = int(rng.integers(6, 23)), int(rng.integers(0, 60))\n        items.append(dict(idx=i, rooms=[int(r) for r in rooms], qline=qline, dist=dist, dstates=dstates,\n                          time=f\"{hh:02d}:{mm:02d}\", A=dstates[0], B=dstates[1]))\n    return items\n\n\ndef log_text(item, domain, D):\n    d = DOMAINS[domain]\n    lines, k = [], 0\n    for li in range(N_LINES):\n        room = item[\"rooms\"][li]\n        if li == item[\"qline\"]:\n            lines.append(f\"- Room {room}: the {d['device']} is {d['states'][D]}.\")\n        else:\n            dev, st = item[\"dist\"][k]\n            lines.append(f\"- Room {room}: the {dev} is {st[item['dstates'][k]]}.\")\n            k += 1\n    return f\"Status log ({item['time']}):\\n\" + \"\\n\".join(lines)\n\n\ndef messages(item, norm, domain, D, R):\n    d = DOMAINS[domain]\n    qroom = item[\"rooms\"][item[\"qline\"]]\n    system = f\"{SYSTEM_PREFIX} {norm_text(norm, domain, R)}\"\n    user = (f\"{log_text(item, domain, D)}\\n\\nWhat state should you report for the {d['device']} in room {qroom}? \"\n            f\"Answer with one word.\")\n    return [{\"role\": \"system\", \"content\": system}, {\"role\": \"user\", \"content\": user}]\n\n\ndef prompt_text(tokenizer, item, norm, domain, D, R):\n    \"\"\"Chat-formatted prompt ending with the assistant prefill; the next token is the answer.\"\"\"\n    text = tokenizer.apply_chat_template(messages(item, norm, domain, D, R), tokenize=False,\n                                         add_generation_prompt=True)\n    return text + PREFILL\n\n\ndef answer_tokens(tokenizer, domain):\n    \"\"\"Token ids of ' state0' and ' state1'. Single tokens are required; otherwise the first tokens are used if\n    they differ (recorded as 'first_token').\"\"\"\n    ids = [tokenizer.encode(\" \" + s, add_special_tokens=False) for s in DOMAINS[domain][\"states\"]]\n    if all(len(x) == 1 for x in ids):\n        return (ids[0][0], ids[1][0]), \"single\"\n    if ids[0][0] != ids[1][0]:\n        return (ids[0][0], ids[1][0]), \"first_token\"\n    raise ValueError(f\"answer tokens of domain {domain} share their first token: {ids}\")\n\n\ndef unit_prompts(tokenizer, norm, domain, n_items, seed):\n    \"\"\"All prompts of one unit (norm x domain): 4 per base item, ordered (item, D, R).\n    Returns texts and arrays item, D, R, y (correct state index = D xor R), A, B, qline.\"\"\"\n    items = make_items(n_items, domain, seed)\n    texts, meta = [], []\n    for it in items:\n        for D in (0, 1):\n            for R in (0, 1):\n                texts.append(prompt_text(tokenizer, it, norm, domain, D, R))\n                meta.append((it[\"idx\"], D, R, D ^ R, it[\"A\"], it[\"B\"], it[\"qline\"]))\n    m = np.array(meta, dtype=np.int64)\n    return texts, dict(item=m[:, 0], D=m[:, 1], R=m[:, 2], y=m[:, 3], A=m[:, 4], B=m[:, 5], qline=m[:, 6])\n", "model_io.py": "\"\"\"LAT-S2 pilot: model loading, batched forward with residual capture, readout gradients for rho_c.\n\nLayer convention (jlens): layer l is the output of block l; the final residual is the output of the last block,\nbefore the final norm. The monitor at layer l reads h_l at the answer position (the last prompt token).\n\"\"\"\nimport math\nimport numpy as np\nimport torch\nimport jlens\n\n\ndef pick_dtype(name=\"auto\"):\n    if name != \"auto\":\n        return getattr(torch, name)\n    if torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8:\n        return torch.bfloat16\n    return torch.float32\n\n\ndef load_model(model_name, dtype=\"auto\", device=None):\n    from transformers import AutoModelForCausalLM, AutoTokenizer\n    device = device or (\"cuda\" if torch.cuda.is_available() else \"cpu\")\n    tok = AutoTokenizer.from_pretrained(model_name)\n    try:\n        hf = AutoModelForCausalLM.from_pretrained(model_name, dtype=pick_dtype(dtype), device_map=device)\n    except (ImportError, ValueError, TypeError):\n        hf = AutoModelForCausalLM.from_pretrained(model_name, dtype=pick_dtype(dtype)).to(device)\n    return wrap(hf, tok)\n\n\ndef wrap(hf, tok):\n    \"\"\"HF model -> jlens LensModel (freezes parameters, finds blocks / final norm / LM head).\"\"\"\n    tok.padding_side = \"left\"\n    if tok.pad_token is None:\n        tok.pad_token = tok.eos_token\n    return jlens.from_hf(hf, tok)\n\n\nclass Capture:\n    \"\"\"One batched forward pass that records, at the last position, the residual after every block, the model's\n    answer contrast, and the gradient of the readout r . h_final[last] with respect to each layer's residual at\n    the last position (w_c(l) = J_c(l)^T r; rho_c needs g_c = w_c . v).\"\"\"\n\n    def __init__(self, lm, tok_pair):\n        self.lm = lm\n        self.L = lm.n_layers\n        self.src_layers = list(range(self.L - 1))\n        U = lm._lm_head.weight\n        self.U2 = U[list(tok_pair)].detach().float()                       # (2, d)\n        gamma = getattr(lm._final_norm, \"weight\", None)\n        g = gamma.detach().float() if gamma is not None else torch.ones(U.shape[1], device=U.device)\n        # readout without the Jacobian of the final RMSNorm: r = gamma * (U[t0] - U[t1]); Qwen2 RMSNorm scales by\n        # weight (no 1 + w offset); for other families check the norm's formula before use.\n        self.r = (g * (self.U2[0] - self.U2[1])).to(U.device)\n\n    def encode(self, texts):\n        enc = self.lm.tokenizer(texts, return_tensors=\"pt\", padding=True, add_special_tokens=False)\n        ids = enc.input_ids.to(self.lm.input_device)\n        mask = enc.attention_mask.to(self.lm.input_device)\n        pos = (mask.cumsum(-1) - 1).clamp(min=0)\n        return ids, mask, pos\n\n    def run(self, texts, grads=True):\n        ids, mask, pos = self.encode(texts)\n        acts = {}\n\n        def make_hook(i):\n            def hook(module, inputs, output):\n                t = output if torch.is_tensor(output) else output[0]\n                if i == 0 and grads:\n                    t.requires_grad_(True)\n                acts[i] = t\n            return hook\n\n        handles = [blk.register_forward_hook(make_hook(i)) for i, blk in enumerate(self.lm.layers)]\n        try:\n            with torch.set_grad_enabled(grads):\n                self.lm._text_module(input_ids=ids, attention_mask=mask, position_ids=pos, use_cache=False)\n                h_final = acts[self.L - 1][:, -1, :]\n                logits2 = torch.nn.functional.linear(self.lm._final_norm(h_final).float(), self.U2)\n                c_out = (logits2[:, 0] - logits2[:, 1]).detach().cpu().numpy()\n                res = np.stack([acts[l][:, -1, :].detach().float().cpu().numpy() for l in range(self.L)], 1)\n                W = None\n                if grads:\n                    s = (h_final.float() @ self.r).sum()\n                    gs = torch.autograd.grad(s, [acts[l] for l in self.src_layers])\n                    W = np.stack([g[:, -1, :].float().cpu().numpy() for g in gs], 1)\n        finally:\n            for h in handles:\n                h.remove()\n        return dict(res=res, c_out=c_out, W=W)\n\n\ndef capture_all(cap, texts, batch_size, grads=True, log=None):\n    out = dict(res=[], c_out=[], W=[])\n    for b0 in range(0, len(texts), batch_size):\n        r = cap.run(texts[b0:b0 + batch_size], grads=grads)\n        for k in out:\n            if r[k] is not None:\n                out[k].append(r[k])\n        if log and (b0 // batch_size) % 10 == 0:\n            log(f\"    batch {b0 // batch_size + 1}/{math.ceil(len(texts) / batch_size)}\")\n    return {k: (np.concatenate(v) if v else None) for k, v in out.items()}\n\n\n@torch.no_grad()\ndef lens_contrasts(lm, lens, res, tok_pair, layers, chunk=256):\n    \"\"\"Lens answer contrast at each layer: unembed(J_l h_l) restricted to the two answer tokens.\n    res: (n, L, d) residuals at the answer position. Returns (n, len(layers)) with NaN where the lens has no J_l.\"\"\"\n    U2 = lm._lm_head.weight[list(tok_pair)].float()\n    dev = U2.device\n    out = np.full((res.shape[0], len(layers)), np.nan)\n    for j, l in enumerate(layers):\n        if l not in lens.jacobians:\n            continue\n        J = lens.jacobians[l].to(dev)\n        for c0 in range(0, res.shape[0], chunk):\n            h = torch.from_numpy(res[c0:c0 + chunk, l]).to(dev)\n            t = h @ J.T\n            z = torch.nn.functional.linear(lm._final_norm(t.to(lm._lm_head.weight.dtype)).float(), U2)\n            out[c0:c0 + chunk, j] = (z[:, 0] - z[:, 1]).cpu().numpy()\n    return out\n\n\n@torch.no_grad()\ndef calibrate_temperatures(lm, lens, texts, layers, max_len=128, skip_first=16,\n                           grid=tuple(np.exp(np.linspace(np.log(0.25), np.log(16), 33)))):\n    \"\"\"One temperature per lens layer: minimises next-token cross-entropy of softmax(lens logits / T) on generic\n    text (positions >= skip_first), as for the LAT-S1 lenses.\"\"\"\n    layers = [l for l in layers if l in lens.jacobians]\n    acts = {l: [] for l in layers}\n    targets = []\n    for text in texts:\n        ids = lm.encode(text, max_length=max_len)\n        if ids.shape[1] <= skip_first + 1:\n            continue\n        with jlens.ActivationRecorder(lm.layers, at=layers) as rec:\n            lm.forward(ids)\n            for l in layers:\n                acts[l].append(rec.activations[l][0, skip_first:-1].float().cpu())\n        targets.append(ids[0, skip_first + 1:])\n    n = int(sum(t.numel() for t in targets))\n    temps = {}\n    if not n:\n        return temps, n\n    dev = lm._lm_head.weight.device\n    for l in layers:\n        J = lens.jacobians[l].to(dev)\n        tot = torch.zeros(len(grid))\n        for a, target in zip(acts[l], targets):\n            logits = lm.unembed(a.to(dev) @ J.T).float()                       # (P, V)\n            for k, T in enumerate(grid):\n                tot[k] += torch.nn.functional.cross_entropy(logits / T, target.to(dev), reduction=\"sum\").cpu()\n        temps[l] = float(grid[int(torch.argmin(tot))])\n        del J\n    return temps, n\n", "measures_s2.py": "\"\"\"LAT-S2 pilot measures, ported from LAT-S1b to natural-language norms.\n\nProbabilities are binary: p = sigmoid(contrast / T), contrast = logit(state0) - logit(state1) at the answer position.\n  Use_R  slope of the model's Delta p on the ideal Delta p over R-twins (both D)          (LAT: on the Bayes filter)\n  Use_D  same over D-twins with R off\n  Seen_k slope of the lens's Delta p on the model's Delta p over the twins of k            (LAT-S1 Seen)\n  rel_R  Seen_R / Seen_D with a joint bootstrap\nClasses follow LAT-S1b: rel_R sighted if the 95% CI lower bound >= 0.8, blind if the upper bound <= 0.5; Seen_R\nsighted if lower >= 0.3, blind if upper <= 0.1; rel_R is defined when Use_R, Use_D >= 0.2 and Seen_D >= 0.1; a unit\ngets a class at a layer when rel_R (main lens), rel_R (control lens) and Seen_R (main lens) agree.\nrho_c: Ostrow et al. Eq. 10 with one source position (the answer position) and lag 0, readout\nr = gamma * (U[s0] - U[s1]) without the final-norm Jacobian: g_c = (J_c^T r) . v_R, rho_c = |E g_c| / E |g_c|,\nv_R the encoding direction of R from regressing h_l on [D, R, A, B, line] (LAT-S1b, coherence.encoding_directions).\nBootstrap: 1000 resamples of base items.\n\"\"\"\nimport warnings\nimport numpy as np\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.pipeline import make_pipeline\nfrom sklearn.preprocessing import StandardScaler\nfrom sklearn.decomposition import PCA\nfrom sklearn.exceptions import ConvergenceWarning\n\nwarnings.filterwarnings(\"ignore\", category=ConvergenceWarning)\nN_BOOT = 1000\nREL_SIGHTED, REL_BLIND = 0.8, 0.5\nSEEN_SIGHTED, SEEN_BLIND = 0.3, 0.1\nUSE_MIN, SEEN_D_MIN = 0.2, 0.1\n\n\ndef sigmoid(x):\n    return 1.0 / (1.0 + np.exp(-x))\n\n\ndef boot_indices(n_items, seed):\n    return np.random.default_rng([seed, 99]).integers(0, n_items, (N_BOOT, n_items))\n\n\ndef ratio(num_item, den_item, B):\n    \"\"\"Ratio of sums over items with bootstrap over items.\"\"\"\n    den = den_item.sum()\n    point = num_item.sum() / den if den > 0 else np.nan\n    dsum = den_item[B].sum(1)\n    bs = num_item[B].sum(1) / np.where(dsum > 0, dsum, np.nan)\n    return float(point), bs\n\n\ndef summ(point, bs):\n    ok = bs[np.isfinite(bs)]\n    ci = [float(np.percentile(ok, 2.5)), float(np.percentile(ok, 97.5))] if len(ok) > 10 else [np.nan, np.nan]\n    return dict(point=float(point), ci=ci)\n\n\nclass Twins:\n    \"\"\"Index of the four prompts of each base item: ix[item, D, R].\"\"\"\n\n    def __init__(self, meta, n_items):\n        self.N = n_items\n        self.ix = np.full((n_items, 2, 2), -1)\n        self.ix[meta[\"item\"], meta[\"D\"], meta[\"R\"]] = np.arange(len(meta[\"item\"]))\n        assert (self.ix >= 0).all()\n\n    def pairs(self, k, fixed=None):\n        \"\"\"(item, idx_hi, idx_lo) for twins of k: k=1 minus k=0. fixed: value of the other factor, or None for both.\"\"\"\n        out = []\n        for v in ((0, 1) if fixed is None else (fixed,)):\n            for i in range(self.N):\n                if k == \"R\":\n                    out.append((i, self.ix[i, v, 1], self.ix[i, v, 0]))\n                else:\n                    out.append((i, self.ix[i, 1, v], self.ix[i, 0, v]))\n        return np.array(out)\n\n    def slope(self, dy, dx, items, B):\n        num = np.bincount(items, weights=dy * dx, minlength=self.N)\n        den = np.bincount(items, weights=dx * dx, minlength=self.N)\n        return ratio(num, den, B)\n\n\ndef ideal_p(meta):\n    \"\"\"Ideal probability of state 0: the correct answer is y = D xor R.\"\"\"\n    return (meta[\"y\"] == 0).astype(float)\n\n\ndef behaviour(meta, c_out, tw, B):\n    p = sigmoid(c_out)\n    ip = ideal_p(meta)\n    res = {}\n    for k, fixed in ((\"R\", None), (\"D\", 0), (\"D_R1\", 1)):\n        kk = \"D\" if k.startswith(\"D\") else \"R\"\n        P = tw.pairs(kk, fixed)\n        pt, bs = tw.slope(p[P[:, 1]] - p[P[:, 2]], ip[P[:, 1]] - ip[P[:, 2]], P[:, 0], B)\n        res[\"Use_\" + k] = summ(pt, bs) | {\"_bs\": bs}\n    correct = (np.sign(c_out) == np.where(meta[\"y\"] == 0, 1, -1))\n    res[\"acc_R0\"] = float(correct[meta[\"R\"] == 0].mean())\n    res[\"acc_R1\"] = float(correct[meta[\"R\"] == 1].mean())\n    return res\n\n\ndef seen(c_lens, temp, c_out, tw, B):\n    \"\"\"Seen_R (R-twins, both D) and Seen_D (D-twins, R off) for one lens at one layer.\"\"\"\n    if not np.isfinite(c_lens).all():\n        return None\n    q = sigmoid(c_lens / temp)\n    p = sigmoid(c_out)\n    out = {}\n    for k, fixed in ((\"R\", None), (\"D\", 0)):\n        P = tw.pairs(k, fixed)\n        pt, bs = tw.slope(q[P[:, 1]] - q[P[:, 2]], p[P[:, 1]] - p[P[:, 2]], P[:, 0], B)\n        out[\"Seen_\" + k] = summ(pt, bs) | {\"_bs\": bs}\n    rp = out[\"Seen_R\"][\"point\"] / out[\"Seen_D\"][\"point\"] if out[\"Seen_D\"][\"point\"] != 0 else np.nan\n    out[\"rel_R\"] = summ(rp, out[\"Seen_R\"][\"_bs\"] / out[\"Seen_D\"][\"_bs\"])\n    return out\n\n\ndef rel_class(rel, beh, s):\n    defined = (beh[\"Use_R\"][\"point\"] >= USE_MIN and beh[\"Use_D\"][\"point\"] >= USE_MIN\n               and s[\"Seen_D\"][\"point\"] >= SEEN_D_MIN)\n    if not defined:\n        return \"undefined\"\n    lo, hi = rel[\"ci\"]\n    return \"sighted\" if lo >= REL_SIGHTED else (\"blind\" if hi <= REL_BLIND else \"none\")\n\n\ndef seen_class(sr):\n    lo, hi = sr[\"ci\"]\n    return \"sighted\" if lo >= SEEN_SIGHTED else (\"blind\" if hi <= SEEN_BLIND else \"none\")\n\n\ndef unit_class(readings):\n    return readings[0] if len(set(readings)) == 1 and readings[0] in (\"sighted\", \"blind\") else \"none\"\n\n\ndef features(meta):\n    pm = lambda x: 2.0 * x - 1.0\n    q = meta[\"qline\"]\n    return np.stack([pm(meta[\"D\"]), pm(meta[\"R\"]), pm(meta[\"A\"]), pm(meta[\"B\"]),\n                     (q == 1).astype(float), (q == 2).astype(float)], 1)\n\n\ndef encoding_direction(X, F, col=1):\n    \"\"\"Coefficient of feature `col` (R) in the least-squares regression of X on F with intercept, normalised.\"\"\"\n    F1 = np.concatenate([F, np.ones((len(F), 1))], 1)\n    coef, *_ = np.linalg.lstsq(F1, np.asarray(X, np.float64), rcond=None)\n    w = coef[col]\n    return w / np.linalg.norm(w)\n\n\ndef rho_c(g):\n    g = np.asarray(g, np.float64)\n    a = np.abs(g).mean()\n    return dict(rho_c=float(abs(g.mean()) / a) if a > 0 else np.nan, a=float(a), Jv=float(abs(g.mean())))\n\n\ndef coherence_layer(res_l, W_l, meta, F):\n    v = encoding_direction(res_l, F)\n    g = W_l.astype(np.float64) @ v\n    out = rho_c(g)\n    out[\"rho_c_R1\"] = rho_c(g[meta[\"R\"] == 1])[\"rho_c\"]\n    out[\"rho_c_R0\"] = rho_c(g[meta[\"R\"] == 0])[\"rho_c\"]\n    return out\n\n\ndef probe_ba(X, y, groups, seed, n_fold=5):\n    \"\"\"Out-of-fold balanced accuracy of a linear probe (standardise, PCA to at most 128 dimensions fitted on the\n    training fold, logistic regression), folds by base item.\"\"\"\n    ug = np.unique(groups)\n    rng = np.random.default_rng([seed, 5])\n    fold_of_item = dict(zip(ug, rng.permutation(len(ug)) % n_fold))\n    folds = np.array([fold_of_item[g] for g in groups])\n    pred = np.zeros_like(y)\n    for f in range(n_fold):\n        tr, te = folds != f, folds == f\n        if len(np.unique(y[tr])) < 2:\n            pred[te] = y[tr][0]\n            continue\n        k = int(min(128, tr.sum() - 1, X.shape[1]))\n        clf = make_pipeline(StandardScaler(), PCA(n_components=k, random_state=seed),\n                            LogisticRegression(C=0.1, max_iter=2000))\n        clf.fit(X[tr], y[tr])\n        pred[te] = clf.predict(X[te])\n    return float(np.mean([np.mean(pred[y == c] == c) for c in np.unique(y)]))\n\n\ndef strip(d):\n    \"\"\"Drop bootstrap arrays before saving.\"\"\"\n    if isinstance(d, dict):\n        return {k: strip(v) for k, v in d.items() if not k.startswith(\"_\")}\n    if isinstance(d, (np.floating,)):\n        return float(d)\n    return d\n", "run_pilot.py": "\"\"\"LAT-S2 pilot runner: lenses, temperatures, per-unit measures, summary and the go / stop rule.\n\nUnit = norm x domain. Monitored layers: quarter, half and three quarters of the depth (layer l = output of block l).\nGo rule (pilot plan): at one monitored layer, among units where the model follows the norm (accuracy >= 0.8 with\nthe norm on and off), at least 3 units are sighted and at least 3 are blind. Everything else is descriptive.\n\"\"\"\nimport json\nimport os\nimport sys\nimport time\nimport numpy as np\nimport torch\nimport jlens\n\nimport norms as NM\nimport model_io as MI\nimport measures_s2 as MS\n\nPRESETS = {\n    \"debug\": dict(model=\"Qwen/Qwen2.5-1.5B-Instruct\", n_items=24, n_lens_prompts=8, n_temp_prompts=4,\n                  dim_batch=\"auto\", batch_size=\"auto\", control=\"half\", dtype=\"auto\", min_gpu_gb=0,\n                  ignore_preflight=True),   # debug checks the code end to end; preflight is still logged\n    \"main\": dict(model=\"Qwen/Qwen2.5-7B-Instruct\", n_items=96, n_lens_prompts=32, n_temp_prompts=8,\n                 dim_batch=\"auto\", batch_size=\"auto\", control=\"community\", dtype=\"auto\", min_gpu_gb=20,\n                 community_repo=\"anicka/jlens-qwen2.5-7b-instruct\",\n                 community_file=\"qwen2.5-7b-instruct_jlens.pt\"),\n}\nCOMMON = dict(domains=[\"light\", \"door\"], norms=list(NM.NORM_NAMES), lens_max_len=128, seed=2026,\n              follow_min=0.8, n_go=3)\n\n\nclass Log:\n    def __init__(self, path):\n        self.path = path\n\n    def __call__(self, msg):\n        line = f\"[{time.strftime('%H:%M:%S', time.gmtime())} UTC] {msg}\"\n        print(line, flush=True)\n        with open(self.path, \"a\") as f:\n            f.write(line + \"\\n\")\n\n\nCORPUS_SOURCES = [(\"Salesforce/wikitext\", \"wikitext-2-raw-v1\", \"train\"),\n                  (\"Salesforce/wikitext\", \"wikitext-103-raw-v1\", \"validation\"),\n                  (\"wikitext\", \"wikitext-2-raw-v1\", \"train\")]\n\n\ndef lens_corpus(n, seed, allow_fallback, log):\n    \"\"\"Generic text for fitting and calibrating the lens: WikiText paragraphs of at least 600 characters\n    (Salesforce/wikitext on the Hugging Face Hub; the older id 'wikitext' as a last resort).\"\"\"\n    errors = []\n    try:\n        from datasets import load_dataset\n    except Exception as e:\n        errors.append(f\"datasets: {type(e).__name__}\")\n        load_dataset = None\n    for repo, conf, split in (CORPUS_SOURCES if load_dataset else []):\n        try:\n            ds = load_dataset(repo, conf, split=split)\n        except Exception as e:\n            errors.append(f\"{repo}/{conf}/{split}: {type(e).__name__}\")\n            continue\n        rng = np.random.default_rng([seed, 11])\n        out, tried = [], 0\n        for i in rng.permutation(len(ds)):\n            t = ds[int(i)][\"text\"].strip()\n            tried += 1\n            if len(t) >= 600 and not t.startswith(\"=\"):\n                out.append(t)\n            if len(out) >= n:\n                break\n        if len(out) >= n:\n            name = f\"{repo} {conf} {split}\"\n            log(f\"lens corpus: {name}, {len(out)} paragraphs (scanned {tried})\")\n            return out, name\n        errors.append(f\"{repo}/{conf}/{split}: only {len(out)} paragraphs\")\n    if not allow_fallback:\n        raise RuntimeError(\"no lens corpus could be loaded: \" + \"; \".join(errors))\n    log(f\"lens corpus: unavailable ({'; '.join(errors)}); fallback text for local tests only\")\n    rng = np.random.default_rng([seed, 12])\n    words = (\"the of a in to and is was for on with as by at from that light door room status report state \"\n             \"log fan window heater gate open closed off sensor hour minute test mode\").split()\n    return [\" \".join(rng.choice(words, 160)) for _ in range(n)], \"fallback\"\n\n\ndef preflight(lm, cfg, log, n_items=8):\n    \"\"\"Behaviour only, before the expensive lens fit: does the model follow each norm (accuracy with R on and off)?\n    The go rule needs at least 2 * n_go classified units, so fewer following units stop the pilot here.\"\"\"\n    rows = []\n    for domain in cfg[\"domains\"]:\n        tok_pair, tok_mode = NM.answer_tokens(lm.tokenizer, domain)\n        cap = MI.Capture(lm, tok_pair)\n        for norm in cfg[\"norms\"]:\n            texts, meta = NM.unit_prompts(lm.tokenizer, norm, domain, n_items, cfg[\"seed\"] + 1)\n            c = MI.capture_all(cap, texts, cfg[\"batch_size\"], grads=False)[\"c_out\"]\n            ok = np.sign(c) == np.where(meta[\"y\"] == 0, 1, -1)\n            a0, a1 = float(ok[meta[\"R\"] == 0].mean()), float(ok[meta[\"R\"] == 1].mean())\n            rows.append(dict(domain=domain, norm=norm, acc_R0=a0, acc_R1=a1, tok_mode=tok_mode,\n                             follows=bool(a0 >= cfg[\"follow_min\"] and a1 >= cfg[\"follow_min\"])))\n            log(f\"preflight {domain}/{norm}: accuracy norm off {a0:.2f}, norm on {a1:.2f}\"\n                f\"{'' if rows[-1]['follows'] else '  (not followed)'}\")\n    return rows\n\n\ndef get_lens(lm, prompts, path, ckpt, cfg, log):\n    if os.path.exists(path):\n        log(f\"lens: loading {os.path.basename(path)}\")\n        return jlens.JacobianLens.load(path)\n    t0 = time.time()\n    log(f\"lens: fitting on {len(prompts)} prompts, dim_batch {cfg['dim_batch']}, max_len {cfg['lens_max_len']}\")\n    # the running-sum checkpoint is large (fp32, n_layers * d_model^2); it lives on the runtime's local disk\n    lens = jlens.fit(lm, prompts, dim_batch=cfg[\"dim_batch\"], max_seq_len=cfg[\"lens_max_len\"],\n                     checkpoint_path=ckpt, checkpoint_every=4)\n    lens.save(path)\n    log(f\"lens: fitted in {(time.time() - t0) / 60:.1f} min -> {os.path.basename(path)}\")\n    return lens\n\n\ndef run_unit(lm, lenses, temps, norm, domain, cfg, monitored, log):\n    tok_pair, tok_mode = NM.answer_tokens(lm.tokenizer, domain)\n    texts, meta = NM.unit_prompts(lm.tokenizer, norm, domain, cfg[\"n_items\"], cfg[\"seed\"])\n    cap = MI.Capture(lm, tok_pair)\n    t0 = time.time()\n    data = MI.capture_all(cap, texts, cfg[\"batch_size\"], grads=True)\n    src = cap.src_layers\n    tw = MS.Twins(meta, cfg[\"n_items\"])\n    B = MS.boot_indices(cfg[\"n_items\"], cfg[\"seed\"])\n    beh = MS.behaviour(meta, data[\"c_out\"], tw, B)\n    F = MS.features(meta)\n    per_layer = {}\n    contrasts = {name: MI.lens_contrasts(lm, lens, data[\"res\"], tok_pair, src) for name, lens in lenses.items()}\n    for j, l in enumerate(src):\n        entry = {}\n        for name in lenses:\n            T = temps[name].get(l)\n            s = MS.seen(contrasts[name][:, j], T, data[\"c_out\"], tw, B) if T else None\n            entry[name] = s\n        readings = []\n        if entry.get(\"main\") and entry.get(\"control\"):\n            readings = [MS.rel_class(entry[\"main\"][\"rel_R\"], beh, entry[\"main\"]),\n                        MS.rel_class(entry[\"control\"][\"rel_R\"], beh, entry[\"control\"]),\n                        MS.seen_class(entry[\"main\"][\"Seen_R\"])]\n        entry[\"readings\"] = readings\n        entry[\"class\"] = MS.unit_class(readings) if readings else \"none\"\n        entry[\"coherence\"] = MS.coherence_layer(data[\"res\"][:, l], data[\"W\"][:, l], meta, F)\n        if l in monitored:\n            X = data[\"res\"][:, l]\n            entry[\"probes\"] = {k: MS.probe_ba(X, meta[k], meta[\"item\"], cfg[\"seed\"]) for k in (\"y\", \"R\", \"D\")}\n        per_layer[str(l)] = entry\n    follows = beh[\"acc_R0\"] >= cfg[\"follow_min\"] and beh[\"acc_R1\"] >= cfg[\"follow_min\"]\n    out = dict(norm=norm, domain=domain, tok_pair=[int(t) for t in tok_pair], tok_mode=tok_mode,\n               n_items=cfg[\"n_items\"], n_prompts=len(texts), follows=bool(follows),\n               behaviour=MS.strip(beh), layers=MS.strip(per_layer), seconds=round(time.time() - t0, 1),\n               example_prompt=texts[1])\n    arrays = dict(c_out=data[\"c_out\"], **{f\"lens_{k}\": v for k, v in contrasts.items()},\n                  **{k: v for k, v in meta.items()},\n                  res_monitored=data[\"res\"][:, monitored].astype(np.float16),\n                  W_monitored=data[\"W\"][:, monitored].astype(np.float16), monitored=np.array(monitored))\n    return out, arrays\n\n\ndef first_readable(r):\n    \"\"\"Earliest layer at which the main lens reads the fact (Seen_D >= 0.1): the first place where the monitor can\n    see the answer at all, and so where a rule applied later would leave it blind (added after the debug run on\n    the 1.5B model, before any 7B data).\"\"\"\n    for l in sorted(int(k) for k in r[\"layers\"]):\n        m = r[\"layers\"][str(l)].get(\"main\")\n        if m and m[\"Seen_D\"][\"point\"] >= MS.SEEN_D_MIN:\n            return l\n    return None\n\n\ndef count_classes(results, layer_of, cfg):\n    cls = {\"sighted\": [], \"blind\": []}\n    for r in results:\n        l = layer_of(r)\n        if l is None:\n            continue\n        e = r[\"layers\"][str(l)]\n        if r[\"follows\"] and e[\"class\"] in cls:\n            cls[e[\"class\"]].append(e[\"coherence\"][\"rho_c\"])\n    go = len(cls[\"sighted\"]) >= cfg[\"n_go\"] and len(cls[\"blind\"]) >= cfg[\"n_go\"]\n    expl = {}\n    if cls[\"sighted\"] and cls[\"blind\"]:\n        from scipy.stats import mannwhitneyu\n        s, b = np.array(cls[\"sighted\"]), np.array(cls[\"blind\"])\n        auc = float(np.mean([(x < y) + 0.5 * (x == y) for x in s for y in b]))\n        expl = dict(auc_rho_c_sighted_lower=auc,\n                    p_one_sided=float(mannwhitneyu(s, b, alternative=\"less\").pvalue))\n    return dict(n_sighted=len(cls[\"sighted\"]), n_blind=len(cls[\"blind\"]), go=bool(go), exploratory=expl)\n\n\ndef layer_entry(e):\n    m = e.get(\"main\") or {}\n    return dict(cls=e[\"class\"], readings=e[\"readings\"], rel_main=m.get(\"rel_R\", {}).get(\"point\"),\n                rel_ctrl=(e.get(\"control\") or {}).get(\"rel_R\", {}).get(\"point\"),\n                seen_R=m.get(\"Seen_R\", {}).get(\"point\"), seen_D=m.get(\"Seen_D\", {}).get(\"point\"),\n                rho_c=e[\"coherence\"][\"rho_c\"], probe_y=e.get(\"probes\", {}).get(\"y\"))\n\n\ndef summarise(results, monitored, cfg):\n    \"\"\"Go rule: at one of the three fixed layers, or at each unit's first readable layer, at least n_go sighted\n    and n_go blind units among the units that follow their norm.\"\"\"\n    verdict = {str(l): count_classes(results, lambda r, l=l: l, cfg) for l in monitored}\n    verdict[\"first_readable\"] = count_classes(results, first_readable, cfg)\n    rows = []\n    for r in results:\n        row = dict(norm=r[\"norm\"], domain=r[\"domain\"], follows=r[\"follows\"],\n                   acc_R0=round(r[\"behaviour\"][\"acc_R0\"], 3), acc_R1=round(r[\"behaviour\"][\"acc_R1\"], 3),\n                   Use_R=round(r[\"behaviour\"][\"Use_R\"][\"point\"], 3))\n        for l in monitored:\n            row[f\"L{l}\"] = layer_entry(r[\"layers\"][str(l)])\n        ls = first_readable(r)\n        row[\"first_readable\"] = dict(layer=ls, **(layer_entry(r[\"layers\"][str(ls)]) if ls is not None else {}))\n        rows.append(row)\n    return dict(monitored=monitored, verdict=verdict, go=any(v[\"go\"] for v in verdict.values()), rows=rows)\n\n\ndef main(mode=\"debug\", out_dir=\"lat_s2_pilot_out\", overrides=None, allow_fallback_corpus=False, lm=None):\n    cfg = {**COMMON, **PRESETS[mode], **(overrides or {})}\n    os.makedirs(out_dir, exist_ok=True)\n    os.makedirs(os.path.join(out_dir, \"units\"), exist_ok=True)\n    ckpt_dir = \"/content/lat_s2_ckpt\" if os.path.isdir(\"/content\") else out_dir\n    os.makedirs(ckpt_dir, exist_ok=True)\n    log = Log(os.path.join(out_dir, \"log.txt\"))\n    jlens.configure_logging()          # per-prompt progress of the lens fit\n    gpu_gb = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0.0\n    if gpu_gb < cfg[\"min_gpu_gb\"]:\n        raise RuntimeError(f\"mode {mode} needs a GPU with at least {cfg['min_gpu_gb']} GB (L4, A100 or G4); \"\n                           f\"this runtime has {gpu_gb:.0f} GB\")\n    if cfg[\"dim_batch\"] == \"auto\":\n        cfg[\"dim_batch\"] = 16 if gpu_gb >= 30 else 8\n    if cfg[\"batch_size\"] == \"auto\":\n        cfg[\"batch_size\"] = 8 if gpu_gb >= 30 else 4\n    log(f\"LAT-S2 pilot, mode {mode}: {json.dumps({k: v for k, v in cfg.items() if k != 'norms'})}\")\n    if lm is None:\n        lm = MI.load_model(cfg[\"model\"], cfg[\"dtype\"])\n    L = lm.n_layers\n    monitored = [L // 4 - 1, L // 2 - 1, 3 * L // 4 - 1]\n    log(f\"model {cfg['model']}: {L} layers, d_model {lm.d_model}, dtype {next(lm._lm_head.parameters()).dtype}; \"\n        f\"monitored layers {monitored}\")\n    pf_path = os.path.join(out_dir, \"preflight.json\")\n    if os.path.exists(pf_path):\n        pf = json.load(open(pf_path))\n    else:\n        pf = preflight(lm, cfg, log)\n        json.dump(pf, open(pf_path, \"w\"), indent=1)\n    n_follow = sum(r[\"follows\"] for r in pf)\n    log(f\"preflight: {n_follow} of {len(pf)} units follow their norm\")\n    if n_follow < 2 * cfg[\"n_go\"] and not cfg.get(\"ignore_preflight\"):\n        log(f\"STOP before the lens fit: the go rule needs at least {2 * cfg['n_go']} followed units\")\n        summary = dict(mode=mode, model=cfg[\"model\"], go=False, stopped=\"preflight\", preflight=pf, config=cfg)\n        json.dump(summary, open(os.path.join(out_dir, \"summary.json\"), \"w\"), indent=1)\n        return summary\n    corpus, corpus_name = lens_corpus(cfg[\"n_lens_prompts\"] + cfg[\"n_temp_prompts\"], cfg[\"seed\"],\n                                      allow_fallback_corpus, log)\n    fit_prompts, temp_prompts = corpus[:cfg[\"n_lens_prompts\"]], corpus[cfg[\"n_lens_prompts\"]:]\n    lenses = {}\n    if cfg[\"control\"] == \"half\":\n        h = len(fit_prompts) // 2\n        lenses[\"main\"] = get_lens(lm, fit_prompts[:h], os.path.join(out_dir, \"lens_main.pt\"),\n                                  os.path.join(ckpt_dir, \"lens_main.ckpt\"), cfg, log)\n        lenses[\"control\"] = get_lens(lm, fit_prompts[h:], os.path.join(out_dir, \"lens_control.pt\"),\n                                     os.path.join(ckpt_dir, \"lens_control.ckpt\"), cfg, log)\n    else:\n        lenses[\"main\"] = get_lens(lm, fit_prompts, os.path.join(out_dir, \"lens_main.pt\"),\n                                  os.path.join(ckpt_dir, \"lens_main.ckpt\"), cfg, log)\n        try:\n            lenses[\"control\"] = jlens.JacobianLens.from_pretrained(cfg[\"community_repo\"],\n                                                                   filename=cfg[\"community_file\"])\n            log(f\"control lens: {cfg['community_repo']} ({lenses['control']})\")\n        except Exception as e:\n            log(f\"control lens: community lens unavailable ({type(e).__name__}: {e}); \"\n                f\"falling back to a second own lens on half of the prompts\")\n            h = len(fit_prompts) // 2\n            lenses[\"control\"] = get_lens(lm, fit_prompts[h:], os.path.join(out_dir, \"lens_control.pt\"),\n                                         os.path.join(ckpt_dir, \"lens_control.ckpt\"), cfg, log)\n    for name, lens in lenses.items():\n        missing = [l for l in monitored if l not in lens.jacobians]\n        if missing:\n            log(f\"WARNING lens {name} has no J for monitored layers {missing}\")\n    tpath = os.path.join(out_dir, \"temperatures.json\")\n    if os.path.exists(tpath):\n        temps = {k: {int(l): t for l, t in v.items()} for k, v in json.load(open(tpath)).items()}\n    else:\n        temps = {}\n        for name, lens in lenses.items():\n            temps[name], n_tok = MI.calibrate_temperatures(lm, lens, temp_prompts, sorted(lens.jacobians),\n                                                           max_len=cfg[\"lens_max_len\"])\n            log(f\"temperatures {name}: monitored {[temps[name].get(l) for l in monitored]} ({n_tok} tokens)\")\n        json.dump(temps, open(tpath, \"w\"), indent=1)\n    results = []\n    units = [(d, n) for d in cfg[\"domains\"] for n in cfg[\"norms\"]]\n    for k, (domain, norm) in enumerate(units):\n        path = os.path.join(out_dir, \"units\", f\"{domain}__{norm}.json\")\n        if os.path.exists(path):\n            results.append(json.load(open(path)))\n            continue\n        out, arrays = run_unit(lm, lenses, temps, norm, domain, cfg, monitored, log)\n        json.dump(out, open(path, \"w\"), indent=1)\n        np.savez_compressed(path.replace(\".json\", \".npz\"), **arrays)\n        results.append(out)\n        cl = [out[\"layers\"][str(l)][\"class\"] for l in monitored]\n        log(f\"unit {k + 1}/{len(units)} {domain}/{norm}: follows {out['follows']} \"\n            f\"(acc {out['behaviour']['acc_R0']:.2f}/{out['behaviour']['acc_R1']:.2f}), classes {cl}, \"\n            f\"rho_c {[round(out['layers'][str(l)]['coherence']['rho_c'], 2) for l in monitored]}, \"\n            f\"{out['seconds']:.0f} s\")\n        if torch.cuda.is_available():\n            torch.cuda.empty_cache()\n    summary = summarise(results, monitored, cfg)\n    summary.update(mode=mode, model=cfg[\"model\"], lens_corpus=corpus_name, config=cfg, preflight=pf,\n                   gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else \"cpu\")\n    json.dump(summary, open(os.path.join(out_dir, \"summary.json\"), \"w\"), indent=1)\n    log(f\"VERDICT: {'GO' if summary['go'] else 'STOP'} | \" +\n        \"; \".join(f\"{'L' + l if l.isdigit() else l}: {v['n_sighted']} sighted, {v['n_blind']} blind\"\n                  for l, v in summary[\"verdict\"].items()))\n    return summary\n\n\nif __name__ == \"__main__\":\n    main(sys.argv[1] if len(sys.argv) > 1 else \"debug\", sys.argv[2] if len(sys.argv) > 2 else \"lat_s2_pilot_out\")\n"}
os.makedirs("/content/lat_s2_pilot", exist_ok=True)
for name, src in CODE.items():
    with open(os.path.join("/content/lat_s2_pilot", name), "w") as f:
        f.write(src)
h = hashlib.sha256()
for name in sorted(CODE):
    h.update(name.encode()); h.update(CODE[name].encode())
CODE_HASH = h.hexdigest()[:16]
assert CODE_HASH == "7188c03ec41d3740", CODE_HASH
print("code hash", CODE_HASH)

In [ ]:
import sys, json
sys.path.insert(0, "/content/lat_s2_pilot")
import run_pilot
OUT = os.path.join(DRIVE_DIR, MODE)
os.makedirs(OUT, exist_ok=True)
open(os.path.join(OUT, "code_hash.txt"), "w").write(CODE_HASH)
summary = run_pilot.main(MODE, OUT)

In [ ]:
s = summary
print("ИТОГ:", "GO – идём в LAT-S2" if s["go"] else "STOP", s.get("stopped", ""))
if s.get("stopped") == "preflight":
    for r in s["preflight"]:
        print(f'{r["domain"]:6s} {r["norm"]:18s} точность без нормы {r["acc_R0"]:.2f}, с нормой {r["acc_R1"]:.2f}')
else:
    for l, v in s["verdict"].items():
        name = f"слой {l}" if l.isdigit() else "первый читаемый слой"
        print(f"{name}: зрячих {v['n_sighted']}, слепых {v['n_blind']}, разведочно {v['exploratory']}")
    print()
    ls = s["monitored"]
    print(f'{"предмет/норма":28s} {"следует":8s} ' + "  ".join(f"слой {l}: класс, rel, Seen_R, rho_c" for l in ls)
          + "  первый читаемый: слой, класс, rel, Seen_R, rho_c")
    for r in s["rows"]:
        cells = []
        for l in ls:
            e = r[f"L{l}"]
            f = lambda x: "  –  " if x is None or x != x else f"{x:5.2f}"
            cells.append(f'{e["cls"]:8s} {f(e["rel_main"])} {f(e["seen_R"])} {f(e["rho_c"])}')
        e = r["first_readable"]
        fr = "  –" if e["layer"] is None else f'{e["layer"]:3d} {e["cls"]:8s} {f(e["rel_main"])} {f(e["seen_R"])} {f(e["rho_c"])}'
        print(f'{r["domain"] + "/" + r["norm"]:28s} {str(r["follows"]):8s} ' + "  ".join(cells) + "  " + fr)
print("\nФайлы на Диске:", OUT)

**Что прислать мне после прогона:** файлы `summary.json` и `log.txt` из папки на Диске (`LAT-S2/pilot/debug` или `LAT-S2/pilot/main`). Остальное там же: линзы, температуры, результаты по каждой норме.